In [1]:
# 1 - impute missing values (age and embarked)

In [2]:
# 2 - one hot encoding (sex and embarked)

In [3]:
# 3 - scaling 

In [4]:
# 4 - feature - selection 

In [5]:
# 5 - DT

In [2]:
import pandas as pd 

In [3]:
df = pd.read_csv('titanic.csv')

In [4]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'],inplace=True)

In [8]:
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['Survived']),df['Survived'],test_size=0.2,random_state=42)

In [7]:
from sklearn.model_selection import train_test_split

In [9]:
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5000,S
733,2,male,23.0,0,0,13.0000,S
382,3,male,32.0,0,0,7.9250,S
704,3,male,26.0,1,0,7.8542,S
813,3,female,6.0,4,2,31.2750,S


In [10]:
y_train.sample(5)

400    1
29     0
194    1
797    1
487    0
Name: Survived, dtype: int64

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
# restarted kernal - you should always try to use the index values rather than the column name 

In [12]:
# 1st step of pipeline
t1 = ColumnTransformer([('impute_age', SimpleImputer(), [2]),('impute_embarked', SimpleImputer(strategy='most_frequent'), [6])], remainder='passthrough')

In [13]:
from sklearn.preprocessing import OneHotEncoder

In [15]:
temp = t1.fit_transform(X_train)
pd.DataFrame(temp).head()

,0,1,2,3,4,5,6
0,45.5,S,1,male,0,0,28.5
1,23.0,S,2,male,0,0,13.0
2,32.0,S,3,male,0,0,7.925
3,26.0,S,3,male,1,0,7.8542
4,6.0,S,3,female,4,2,31.275


In [18]:
t2 = ColumnTransformer([('ohe_sex_embarked',OneHotEncoder(sparse_output=False, handle_unknown='ignore'),[1,3])], remainder='passthrough')

In [21]:
from sklearn.preprocessing import MinMaxScaler

In [22]:
# after upar vale dono transformations we will have 10 cols in total

In [24]:
t3 = ColumnTransformer([('scale', MinMaxScaler(), [5,9])], remainder='passthrough')

In [25]:
from sklearn.feature_selection import SelectKBest,chi2

In [26]:
t4 = SelectKBest(score_func=chi2, k=8)

In [27]:
from sklearn.tree import DecisionTreeClassifier

In [28]:
t5 = DecisionTreeClassifier()

In [29]:
from sklearn.pipeline import Pipeline,make_pipeline

In [30]:
pipe = Pipeline([('trf1',t1),('trf2',t2),('trf3',t3),('trf4',t4),('trf5',t5)])

In [31]:
# if we do only preprocessing vale steps through pipeline we we should call fit_transform

In [32]:
pipe.fit(X_train, y_train)

,steps,"[('trf1', ...), ('trf2', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('impute_age', ...), ('impute_embarked', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [33]:
y_pred = pipe.predict(X_test)

In [34]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.7877094972067039

In [35]:
pipe.named_steps

{'trf1': ColumnTransformer(remainder='passthrough',
                   transformers=[('impute_age', SimpleImputer(), [2]),
                                 ('impute_embarked',
                                  SimpleImputer(strategy='most_frequent'),
                                  [6])]),
 'trf2': ColumnTransformer(remainder='passthrough',
                   transformers=[('ohe_sex_embarked',
                                  OneHotEncoder(handle_unknown='ignore',
                                                sparse_output=False),
                                  [1, 3])]),
 'trf3': ColumnTransformer(remainder='passthrough',
                   transformers=[('scale', MinMaxScaler(), [5, 9])]),
 'trf4': SelectKBest(k=8, score_func=<function chi2 at 0x0000026797A271A0>),
 'trf5': DecisionTreeClassifier()}

In [36]:
import pickle 

In [37]:
pickle.dump(pipe , open('pipe.pkl', 'wb'))